# 05e — Recuperación de la no respuesta de ingresos por cotas

Diagnóstico, **no escribe capas ni cambia el universo**: solo `data/meta/no_respuesta_cotas_*.csv` y una fila
en `data/meta/corridas_no_respuesta.csv`. La variable objetivo no se imputa.

**Idea.** Un hogar con no respuesta igual tiene ingresos declarados (o no imputados). Su suma es una **cota
inferior** del ingreso real: si ya alcanza la línea de pobreza, el hogar es `seguro_no_pobre` sin suponer nada
sobre lo que falta. Si no alcanza, queda `indeterminado`; si no declara nada, `sin_ingreso_declarado`.

- **Era regular** (2016T2–2025T4): no respuesta = `ingreso_no_declarado` (`PONDIH == 0`, `DECCFR == 12`). El
  `ITF` de esos hogares vale 0, así que la cota se arma desde la base individual (`src/no_respuesta_cotas.py`):
  `P21` (o, si es `-9`, el mayor de `PP08D1+PP08F1+PP08F2`, `PP06C`, `PP06D`), `PP08J1–3`, `TOT_P12` y `T_VI`
  (o, si es `-9`, la suma de sus `V*_M` declarados), con `-9` contando 0.
- **Era histórica** (2003T3–2015T2): no respuesta = `IDIMPH` con alguna imputación. Cota = `ITF −
  monto_imputado_max` (05d): el ingreso no imputado, que no depende de que el hot-deck de INDEC sea correcto.

**Candado del holdout.** Los conteos de pares se publican por trimestre solo para train y embargo; del holdout
(origen ≥ 2023T1) solo el total agregado, sin tasas de la etiqueta. Toda tasa de `pobre_th` es de train.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import no_respuesta_cotas as nc
import panel
import pipeline_meta
from armonizacion import trimestres_publicados
from features import CLAVE_PAR

ARMONIZADO = REPO / "data" / "01_armonizado"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
META = REPO / "data" / "meta"
CORRIDAS = META / "corridas_no_respuesta.csv"
HOLDOUT_DESDE = (2023, 1)
TRIMESTRES_HIST = [t for t in trimestres_publicados() if t <= (2015, 2)]
TRIMESTRES_REG = [t for t in trimestres_publicados() if t >= (2016, 2)]
CLAVE_HOGAR_TRIMESTRE = ["CODUSU", "NRO_HOGAR", "AGLOMERADO", "ANO4", "TRIMESTRE"]
COLUMNAS_HOGAR = [*CLAVE_HOGAR_TRIMESTRE, "con_no_respuesta", "clase", "ratio_cota", "pobre"]
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
T0 = time.time()

## 1. Cota por hogar, trimestre por trimestre

Chequeos por trimestre (se cuentan, no se corrigen): en la regular, personas con algún `-9`, `P47T` en `-9`,
`P47T` positivo con algún componente en `-9` (un `P47T` parcial), `-9` en hogares declarantes, declarantes
cuya cota no reproduce el `ITF`, y hogares con no respuesta sin ningún `-9` en los montos. En la histórica,
fracción máxima en `NA` o igual a 1, monto máximo mayor que el `ITF` y hogares `seguro_no_pobre` que INDEC
deja como pobres (tiene que dar 0: el ITF imputado nunca es menor que la parte no imputada).

In [ ]:
hogares_por_era, chequeos_por_era = {}, {}
for era, trimestres, funcion in [("regional_2016", TRIMESTRES_REG, nc.clases_cota_regular),
                                  ("historica", TRIMESTRES_HIST, nc.clases_cota_historica)]:
    partes, chequeos = [], []
    for anio, trimestre in trimestres:
        hogares, chequeo = funcion(anio, trimestre, ARMONIZADO, DISTANCIA)
        partes.append(hogares[COLUMNAS_HOGAR])
        chequeos.append(chequeo)
    hogares_por_era[era] = pd.concat(partes, ignore_index=True).assign(era=era)
    chequeos_por_era[era] = pd.DataFrame(chequeos)
    print(f"{era}: {len(trimestres)} trimestres, {len(hogares_por_era[era]):,} hogares ({time.time() - T0:.0f}s)")

hogares = pd.concat(hogares_por_era.values(), ignore_index=True)
assert not hogares.duplicated(CLAVE_HOGAR_TRIMESTRE).any(), "clave de hogar duplicada dentro de un trimestre"
chequeos_por_era["regional_2016"].to_csv(META / "no_respuesta_cotas_chequeos_regular.csv", index=False)
chequeos_por_era["historica"].to_csv(META / "no_respuesta_cotas_chequeos_historica.csv", index=False)
for era, tabla in chequeos_por_era.items():
    print(f"\n{era} -- totales de los chequeos:")
    display(tabla.drop(columns=["ANO4", "TRIMESTRE"]).sum().to_frame("total").T)

## 2. Hogares por clase de cota y trimestre

Son conteos de corte transversal (como las tasas de pobreza por trimestre que ya publica la 02b), no pares:
se muestran todos los trimestres.

In [ ]:
hogares_trimestre = (hogares.groupby(["era", "ANO4", "TRIMESTRE", "clase"]).size().unstack("clase", fill_value=0)
                     .reindex(columns=list(nc.CLASES_COTA), fill_value=0))
hogares_trimestre["n_hogares"] = hogares_trimestre.sum(axis=1)
hogares_trimestre["n_no_respuesta"] = hogares_trimestre["n_hogares"] - hogares_trimestre["declarado"]
hogares_trimestre["pct_no_respuesta"] = 100 * hogares_trimestre["n_no_respuesta"] / hogares_trimestre["n_hogares"]
hogares_trimestre["pct_recuperable_seguro_no_pobre"] = (100 * hogares_trimestre["seguro_no_pobre"]
                                                         / hogares_trimestre["n_no_respuesta"])
hogares_trimestre = hogares_trimestre.reset_index()
hogares_trimestre.to_csv(META / "no_respuesta_cotas_hogares_trimestre.csv", index=False)

resumen_hogares = hogares[hogares["con_no_respuesta"]].groupby("era")["clase"].value_counts(normalize=True).unstack()
resumen_hogares.insert(0, "n_no_respuesta", hogares[hogares["con_no_respuesta"]].groupby("era").size())
resumen_hogares.insert(1, "pct_hogares_no_respuesta", 100 * hogares.groupby("era")["con_no_respuesta"].mean())
display((resumen_hogares * [1, 1, *[100] * (resumen_hogares.shape[1] - 2)]).round(1))
display(hogares_trimestre[["era", "ANO4", "TRIMESTRE", "n_hogares", "pct_no_respuesta", *nc.CLASES_COTA[1:],
                           "pct_recuperable_seguro_no_pobre"]].round(1).to_string(index=False))

Qué tan arriba de la línea queda la cota de los `seguro_no_pobre`, y qué tan cerca de la línea quedan los
`indeterminados` (cota / línea). Un indeterminado con cota de 0,9 está "casi recuperado"; uno con 0,1 no dice nada.

In [ ]:
cuantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
distribucion_cota = (hogares[hogares["clase"].isin(["seguro_no_pobre", "indeterminado"])]
                     .groupby(["era", "clase"])["ratio_cota"].quantile(cuantiles).unstack())
distribucion_cota.columns = [f"p{int(q * 100)}" for q in cuantiles]
distribucion_cota.reset_index().to_csv(META / "no_respuesta_cotas_distribucion_ratio.csv", index=False)
distribucion_cota.round(3)

## 3. Pares recuperables `t → t+h`

Universo: los pares de `data/05_trayectoria_distancia/` con `identidad_confirmada`. "Válido hoy" es el criterio
principal (`imputacion_excluye`): `par_valido_c` en la histórica, `par_valido_simetrica` en la regular (donde
coinciden). Un par se recupera para un objetivo si hoy no es válido, al menos una punta tiene no respuesta y las
dos puntas tienen `pobre` conocido (declarado o `seguro_no_pobre`):

- **entrada** (`pobre_t == False`): recuperable por `t` (`t` seguro_no_pobre, `t+h` declarado), por `t+h`
  (`t+h` seguro_no_pobre → `pobre_th = False`) o por ambas.
- **permanencia** (`pobre_t == True`): solo por `t+h`, porque la cota nunca prueba que un hogar es pobre: `t`
  tiene que estar declarado, y el par recuperado siempre sale de la pobreza.
- **trayectoria**: ninguno. La cota da un piso del ratio, no su valor; la variación queda sin definir.

La vía importa: lo recuperado por `t+h` entra **solo con la etiqueta 0**, así que la recuperación depende del
resultado (selección sobre Y). Lo recuperado por `t` depende solo de la foto en `t`.

In [ ]:
COLUMNAS_05 = [*CLAVE_PAR, "ANO4_th", "TRIMESTRE_th", "AGLOMERADO_t", "AGLOMERADO_th", "era_t", "identidad_confirmada",
               "pobre_t", "pobre_th", "ratio_linea_pobreza_t", "ratio_linea_pobreza_th", "par_valido_simetrica",
               "par_valido_c"]


def leer_pares_05(h: int) -> pd.DataFrame:
    partes = []
    for ruta in sorted((TRAYECTORIA / f"pares_h{h}").glob("ANO4=*/TRIMESTRE=*/data.parquet")):
        disponibles = pq.read_schema(ruta).names
        partes.append(pd.read_parquet(ruta, columns=[c for c in COLUMNAS_05 if c in disponibles]))
    pares = pd.concat(partes, ignore_index=True)
    historica = pares["era_t"] == "historica"
    pares["valido_actual"] = np.where(historica, pares["par_valido_c"].astype("boolean").fillna(False),
                                      pares["par_valido_simetrica"].astype("boolean").fillna(False)).astype(bool)
    pares["particion"] = panel.particion_train_holdout(pares, HOLDOUT_DESDE)
    return pares


def con_clases(pares: pd.DataFrame) -> pd.DataFrame:
    for lado in ("t", "th"):
        clases = hogares[[*CLAVE_HOGAR_TRIMESTRE, "clase"]].rename(columns={
            "NRO_HOGAR": f"NRO_HOGAR_{lado}", "AGLOMERADO": f"AGLOMERADO_{lado}", "ANO4": f"ANO4_{lado}",
            "TRIMESTRE": f"TRIMESTRE_{lado}", "clase": f"clase_{lado}"})
        pares = pares.merge(clases, on=["CODUSU", f"NRO_HOGAR_{lado}", f"AGLOMERADO_{lado}", f"ANO4_{lado}",
                                        f"TRIMESTRE_{lado}"], how="left", validate="many_to_one")
    return pares


pares_por_h = {}
for h in (1, 4):
    pares = con_clases(leer_pares_05(h))
    sin_clase = pares[["clase_t", "clase_th"]].isna().any(axis=1)
    print(f"h={h}: {len(pares):,} pares, {int(sin_clase.sum())} sin clase en alguna punta")
    pares_por_h[h] = pd.concat([pares, nc.recuperacion_pares(pares)], axis=1)

In [ ]:
def contar(g: pd.DataFrame) -> pd.Series:
    identidad = g["identidad_confirmada"].astype("boolean").fillna(False).astype(bool)
    con_nr = (g["clase_t"] != "declarado") | (g["clase_th"] != "declarado")
    perdido = identidad & ~g["valido_actual"] & con_nr
    pobre_t = g["pobre_t"].astype("boolean")
    fila = {
        "n_pares_identidad": int(identidad.sum()),
        "n_validos_trayectoria": int(g["valido_actual"].sum()),
        "n_validos_entrada": int((g["valido_actual"] & (pobre_t == False).fillna(False)).sum()),  # noqa: E712
        "n_validos_permanencia": int((g["valido_actual"] & (pobre_t == True).fillna(False)).sum()),  # noqa: E712
        "n_perdidos_por_ingreso": int(perdido.sum()),
        "n_recuperados_trayectoria": 0,
    }
    for objetivo in ("entrada", "permanencia"):
        recupera = g[f"recupera_{objetivo}"]
        fila[f"n_recuperados_{objetivo}"] = int(recupera.sum())
        for via in ("t", "th", "ambas"):
            fila[f"n_recuperados_{objetivo}_via_{via}"] = int((recupera & (g["via"] == via)).sum())
    return pd.Series(fila)


filas_trimestre, filas_resumen = [], []
for h, pares in pares_por_h.items():
    pares = pares.assign(h=h, era=pares["era_t"])
    no_holdout = pares[pares["particion"] != "holdout"]
    filas_trimestre.append(no_holdout.groupby(["era", "h", "particion", "ANO4_t", "TRIMESTRE_t"]).apply(contar))
    filas_resumen.append(pares.groupby(["era", "h", "particion"]).apply(contar))
pares_trimestre = pd.concat(filas_trimestre).reset_index()
resumen_pares = pd.concat(filas_resumen).reset_index()
pares_trimestre.to_csv(META / "no_respuesta_cotas_pares_trimestre.csv", index=False)
resumen_pares.to_csv(META / "no_respuesta_cotas_pares_resumen.csv", index=False)

vista = resumen_pares.copy()
for objetivo in ("entrada", "permanencia"):
    vista[f"pct_suma_{objetivo}"] = 100 * vista[f"n_recuperados_{objetivo}"] / vista[f"n_validos_{objetivo}"]
display(vista.round(1).to_string(index=False))

## 4. Criterios de calidad (solo train)

**(a) Selección sobre el resultado.** Prevalencia de `pobre_th` en train con los pares válidos hoy (en los
recuperados, la etiqueta conocida por cota, `pobre_th_cota`), sumando todo lo recuperado, y sumando solo lo recuperado por `t` (la vía que no depende del resultado). Si la prevalencia se
mueve al sumar las vías por `t+h`, esa recuperación sesga la etiqueta por construcción.

In [ ]:
filas = []
for h, pares in pares_por_h.items():
    train = pares[pares["particion"] == "train"]
    for objetivo, valor_pobre_t in (("entrada", False), ("permanencia", True)):
        en_poblacion = (train["pobre_t"].astype("boolean") == valor_pobre_t).fillna(False).astype(bool)
        validos = train[train["valido_actual"] & en_poblacion]
        recuperados = train[train[f"recupera_{objetivo}"]]
        for era in ("historica", "regional_2016"):
            v, r = validos[validos["era_t"] == era], recuperados[recuperados["era_t"] == era]
            if r.empty and v.empty:
                continue
            r_t = r[r["via"] == "t"]
            y_v = v["pobre_th"].astype(float)
            y_r, y_rt = (x["pobre_th_cota"].astype(float) for x in (r, r_t))
            filas.append({
                "era": era, "h": h, "objetivo": objetivo, "n_validos": len(v), "n_recuperados": len(r),
                "n_recuperados_via_t": len(r_t),
                "tasa_validos": y_v.mean(), "tasa_recuperados": y_r.mean() if len(r) else np.nan,
                "tasa_recuperados_via_t": y_rt.mean() if len(r_t) else np.nan,
                "tasa_validos_mas_todo": pd.concat([y_v, y_r]).mean(),
                "tasa_validos_mas_via_t": pd.concat([y_v, y_rt]).mean(),
            })
sesgo = pd.DataFrame(filas)
sesgo["cambio_pp_todo"] = 100 * (sesgo["tasa_validos_mas_todo"] - sesgo["tasa_validos"])
sesgo["cambio_pp_via_t"] = 100 * (sesgo["tasa_validos_mas_via_t"] - sesgo["tasa_validos"])
sesgo.to_csv(META / "no_respuesta_cotas_sesgo_prevalencia.csv", index=False)
sesgo.round(4)

**(b) Perfil de lo recuperado por `t`.** Por construcción son hogares con al menos una línea de ingreso declarado:
se comparan contra los pares válidos de entrada con ratio en `t` ≥ 1 y ≥ 1,5 (los seguro_no_pobre tienen ratio
real ≥ su cota ≥ 1). Si la tasa de entrada de lo recuperado se parece a la de los válidos de ratio alto, la vía `t`
agrega hogares del mismo tipo que ya están; si es mucho mayor, la no respuesta en `t` anticipa deterioro.

In [ ]:
filas = []
for h, pares in pares_por_h.items():
    train = pares[pares["particion"] == "train"]
    entrada_valida = train[train["valido_actual"] & (train["pobre_t"].astype("boolean") == False).fillna(False)]  # noqa: E712
    via_t = train[train["recupera_entrada"] & (train["via"] == "t")]
    for era in ("historica", "regional_2016"):
        ev, vt = entrada_valida[entrada_valida["era_t"] == era], via_t[via_t["era_t"] == era]
        ratio = ev["ratio_linea_pobreza_t"].astype(float)
        filas.append({"era": era, "h": h,
                      "tasa_entrada_recuperados_via_t": vt["pobre_th_cota"].astype(float).mean(), "n_via_t": len(vt),
                      "tasa_entrada_validos_ratio_ge_1": ev.loc[ratio >= 1, "pobre_th"].astype(float).mean(),
                      "tasa_entrada_validos_ratio_ge_1_5": ev.loc[ratio >= 1.5, "pobre_th"].astype(float).mean(),
                      "tasa_entrada_validos_ratio_1_a_1_5": ev.loc[(ratio >= 1) & (ratio < 1.5), "pobre_th"].astype(float).mean()})
perfil_via_t = pd.DataFrame(filas)
perfil_via_t.to_csv(META / "no_respuesta_cotas_perfil_via_t.csv", index=False)
perfil_via_t.round(4)

**(c) Coherencia en la histórica.** Un `seguro_no_pobre` histórico nunca puede ser pobre con el ITF imputado de
INDEC (el ITF incluye la parte no imputada). El chequeo `n_seguro_no_pobre_pobre_indec` de la sección 1 tiene que
dar 0; en la regular, `n_declarantes_cota_distinta_itf` tiene que dar 0 (la regla reproduce el `ITF` donde no
falta nada).

In [ ]:
coherencia = pd.Series({
    "historica_seguro_no_pobre_pobre_indec": int(chequeos_por_era["historica"]["n_seguro_no_pobre_pobre_indec"].sum()),
    "regular_declarantes_cota_distinta_itf": int(chequeos_por_era["regional_2016"]["n_declarantes_cota_distinta_itf"].sum()),
    "regular_p47t_parcial": int(chequeos_por_era["regional_2016"]["n_p47t_parcial"].sum()),
    "regular_no_respuesta_sin_menos9": int(chequeos_por_era["regional_2016"]["n_no_respuesta_sin_menos9"].sum()),
    "historica_monto_max_mayor_itf": int(chequeos_por_era["historica"]["n_monto_max_mayor_itf"].sum()),
})
coherencia.to_frame("n")

In [ ]:
pipeline_meta.registrar_corrida(
    CORRIDAS, "05e_no_respuesta_cotas", {"HOLDOUT_DESDE": HOLDOUT_DESDE, "criterio_valido": "imputacion_excluye"},
    n_trimestres=len(TRIMESTRES_HIST) + len(TRIMESTRES_REG), n_filas=0, duracion_segundos=time.time() - T0,
    commit_git=pipeline_meta.obtener_commit_git(REPO))
print(f"listo en {time.time() - T0:.0f}s")